In [ ]:
import os
import rasterio
import tensorflow as tf
import xarray as xr
import rioxarray as rxr
import geopandas as gpd
from src.config import *
from tensorflow.keras import Sequential
from tensorflow.keras import layers,models
import matplotlib.pyplot as plt
import laspy
import numpy as np

In [ ]:
data_dir = '/home/ajai-krishna/work/GEO_AI/data/outputs/segmented_data/'
data = [os.path.join(data_dir,i) for i in os.listdir(data_dir) if i.endswith('.tif')]
data

In [ ]:
patches = []

for file in os.listdir(data_dir):

    if file.endswith(".tif"):

        path = os.path.join(data_dir, file)

        with rasterio.open(path) as src:
            img = src.read(1)

        patches.append(img)

patches = np.array(patches)

print("Dataset shape:", patches.shape)

In [ ]:
img = da.squeeze().values
print(img.shape)

In [ ]:
img = img.astype(np.float32)

img = (img - np.nanmin(img)) / (np.nanmax(img) - np.nanmin(img))

In [ ]:

def create_patches(image, patch_size=256):
    patches = []
    h, w = image.shape
    
    for i in range(0, h, patch_size):
        for j in range(0, w, patch_size):

            patch = image[i:i+patch_size, j:j+patch_size]

            if patch.shape == (patch_size, patch_size):
                patches.append(patch)

    return np.array(patches)

In [ ]:
patches = create_patches(img, 256)

print(patches.shape)

In [ ]:
patches = np.expand_dims(patches, axis=-1)

print(patches.shape)

In [ ]:
patches = patches.astype("float32")
patches = (patches - patches.min()) / (patches.max() - patches.min())

In [ ]:
labels = np.random.rand(patches.shape[0], 1)

In [ ]:
labels.shape

In [ ]:
model = models.Sequential([
    layers.Conv2D(16, (3,3), activation='relu', padding='same', input_shape=(256,256,1)),
    layers.MaxPooling2D((2,2)),
    
    layers.Conv2D(32, (3,3), activation='relu', padding='same'),
    layers.MaxPooling2D((2,2)),
    
    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dense(2, activation='softmax'),
    layers.Dense(1, activation='sigmoid')# 2 classes
])

In [ ]:
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [ ]:
model.fit(
    patches,
    labels,
    batch_size=16,
    epochs=10,
    validation_split=0.2
)

In [ ]:

# model = models.Sequential()

# model.add(layers.Conv2D(
#     filters=16,
#     kernel_size=(3,3),
#     activation='relu',
#     input_shape=(32,32,1)
# ))

# model.add(layers.MaxPooling2D(pool_size=(2,2)))

# model.add(layers.GlobalAveragePooling2D())

# model.add(layers.Dense(20, activation='softmax'))

# model.compile(
#     optimizer='adam',
#     loss='categorical_crossentropy',
#     metrics=['accuracy']
# )

# model.summary()

In [ ]:
model.fit(
    patches,
    labels,
    batch_size=16,
    epochs=10,
    validation_split=0.2
)